# 도구 (Tools)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

os.getenv("OPENAI_API_KEY")

# 도구 정의

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a:int, b:int):
    """
    두 숫자를 곱합니다.
    """
    return a * b

@tool
def add(a:int, b:int):
    """
    두 숫자를 더합니다.
    """
    return a + b

tools = [multiply, add]

In [ ]:
multiply

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model = "gpt-5-nano",
    temperature = 0
)

llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langchain_core.messages import SystemMessage
from langchain.messages import (HumanMessage, AIMessage, SystemMessage, ToolMessage)

# tuple
messages = [
    ("system", "당신은 친절한 수학 보조입니다."),
    ("user","3 더하기 5는 얼마야?")
]

# dict
messages = [
    {"role":"system","content":"당신은 친철한 수학 보조입니다. 반드시 도구를 호출해서 대답하시오."},
    {"role":"user","content":"3 더하기 5는 얼마야?"}
]


In [ ]:
response = llm_with_tools.invoke(messages)
response

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = {"multiply": multiply, "add": add}[tool_call["name"]]
    tool_output = selected_tool.invoke(tool_call["args"])
    print(tool_output)

    messages.append(ToolMessage(
        tool_call_id=tool_call["id"], 
        content=str(tool_output)
    ))

In [ ]:
messages

In [ ]:
# messages 초기화 (질문 내용 다시 설정)
messages = [
    {"role":"system","content":"당신은 친철한 수학 보조입니다. 반드시 도구를 호출해서 대답하시오."},
    {"role":"user","content":"3 더하기 5는 얼마야?"}
]

# 1. AI에게 질문 보내기 (도구 호출 요청 받기)
response = llm_with_tools.invoke(messages)
messages.append(response) # 대화 기록에 추가

# 2. 도구 실행 정보 추출 및 실행
tool_call = response.tool_calls[0]
tool_name = tool_call["name"]
tool_args = tool_call["args"]

selected_tool = {"multiply": multiply, "add": add}[tool_name]
result_value = selected_tool.invoke(tool_args)

# 3. 도구 실행 결과 메시지 생성 및 추가
tool_message = ToolMessage(
    tool_call_id=tool_call['id'],
    content=str(result_value),
    name=tool_name
)
messages.append(tool_message)

# 4. 최종 답 얻기
final_response = llm_with_tools.invoke(messages)
final_response

## DuckDuckgo Search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import DuckDuckGoSearchResults

web_search = DuckDuckGoSearchRun()
web_search.invoke("오바마 대통령의 풀네임을 알려줘?")

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model = "gpt-5-nano",
    temperature = 0
)

llm_with_tools = llm.bind_tools([web_search])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

messages = (
    [
        ("system","""당신은 웹 검색을 통해서 실시간 저보를 얻을 수 있습니다."""),
        ("user", "현재 엔비디아의 주가를 검색해줘")
    ]
)

response = llm_with_tools.invoke(messages)

In [ ]:
dict(response)

# ToolNode
- Graph Node에서 Tool 호출하는 기능을 가지고 있는 노드

In [ ]:
tools = [web_search]
tools_by_name = {tool.name: tool for tool in tools}

In [ ]:
from langchain.messages import ToolMessage

# llm_with_tools = llm.bind_tools(tools)
def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model = "gpt-5-nano",
    temperature = 0
)

llm_with_tools = llm.bind_tools([web_search])

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages] #add_messages 리듀서

#노드
def chatbot(state: ChatState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

- 테스트

In [ ]:
test_state = {
    "messages": [("user", "'duckduckgo_search'를 사용해서 서울에 대해서 검색해줘")]
}

response = chatbot(test_state)
response

## 그래프 구성

In [ ]:
from langgraph.graph import StateGraph, START

workflow = StateGraph(ChatState)
workflow.add_node("chatbot", chatbot)
workflow.add_node("tools",tool_node)

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END

def should_continue(state: ChatState) -> Literal["tools", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tools"

    # Otherwise, we stop (reply to the user)
    return END

In [ ]:
workflow.add_conditional_edges(
    "chatbot", should_continue, 
    {
        "tools" : "tools",
        END : END
    }
)
workflow.add_edge(START,"chatbot")
workflow.add_edge("tools","chatbot")

graph = workflow.compile()

In [ ]:
from IPython.display import Image, display

png_bytes = graph.get_graph().draw_mermaid_png()

display(Image(data=png_bytes))

In [ ]:
response = graph.invoke(
    {
        "messages":[("user","LangGraph에 대해서 설명해줘")]
    }
)
response

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

## ToolNode, tools_condition

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition

from typing import Literal
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ChatState)
workflow.add_node("chatbot", chatbot)
workflow.add_node("tools",ToolNode(tools=[web_search]))
workflow.add_conditional_edges(
    "chatbot",
    tools_condition
)
workflow.add_edge(START, "chatbot")
workflow.add_edge("tools","chatbot")
graph = workflow.compile()

In [ ]:
# 마크다운 코드를 직접 표시한다. 더 선명하게 보인다.
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = graph.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))